# 06 - Modelos

**Objetivo:** tentar bater o baseline ("semana que vem = esta semana", MAE 27,9 / RMSE 62,7) e escolher o modelo final.

Faço isso em 5 passos, acrescentando **uma coisa por vez** e medindo sempre contra o baseline, nas mesmas semanas e com as mesmas regras do `05_baseline` (janela móvel anual de 2019 a 2026, só semanas fechadas, MAE e RMSE em casos). Assim, se o erro cair, dá pra saber exatamente o que fez ele cair.

| passo | o que entra |
|---|---|
| 1 | regressão linear com os casos das 4 semanas anteriores |
| 2 | + época do ano |
| 3 | + alvo em log |
| 4 | + clima |
| 5 | um modelo não linear (Gradient Boosting) |

As regras de avaliação estão em `src/avaliacao.py`, pra todos os modelos serem avaliados exatamente do mesmo jeito.

In [1]:
import sys

import pandas as pd
from sklearn.linear_model import LinearRegression

# Deixa o Python achar a pasta src/ (que fica um nível acima de notebooks/), igual um import relativo no Node
sys.path.append("..")
from src.avaliacao import ANOS_TESTE, erros_por_ano, separar_treino_teste

dengue = pd.read_csv("../data/processed/dengue_clima_semanal.csv", parse_dates=["data_iniSE"])
dengue["ano"] = dengue["SE"] // 100

print("Em ordem:", dengue["data_iniSE"].is_monotonic_increasing)
print("Sempre 7 dias entre semanas:", (dengue["data_iniSE"].diff().dropna() == pd.Timedelta(days=7)).all())

Em ordem: True
Sempre 7 dias entre semanas: True


## Passo 1: regressão linear com os lags de casos

O primeiro modelo é uma regressão linear: ela aprende um peso pra cada um dos casos das 4 semanas anteriores (lags 1 a 4), e a previsão é a soma de peso × lag. Escolhi ela pra começar porque dá pra explicar cada peso e porque ela consegue prever valores maiores do que viu no treino. Modelos de árvore não conseguem: no teste de 2022, o treino tinha no máximo 620 casos numa semana, e o pico foi de 1.581.

Em cada ano de teste, treino um modelo novo só com os anos anteriores.

In [2]:
# Features do passo 1: os casos das 4 semanas anteriores (lag 1 a 4)
# casos_lag1 = semana anterior, casos_lag2 = duas semanas antes, ...
# f"casos_lag{k}" é uma f-string, igual a template literal `casos_lag${k}` no JS
FEATURES_LAGS = []
for k in range(1, 5):
    dengue[f"casos_lag{k}"] = dengue["casos_est"].shift(k)
    FEATURES_LAGS.append(f"casos_lag{k}")

# O baseline é o próprio lag 1
dengue["previsao_ingenua"] = dengue["casos_lag1"]

dengue[["data_iniSE", "casos_est"] + FEATURES_LAGS].head(6)

,data_iniSE,casos_est,casos_lag1,casos_lag2,casos_lag3,casos_lag4
0,2015-01-04,50.0,NaN,NaN,NaN,NaN
1,2015-01-11,34.0,50.0,NaN,NaN,NaN
2,2015-01-18,51.0,34.0,50.0,NaN,NaN
3,2015-01-25,46.0,51.0,34.0,50.0,NaN
4,2015-02-01,80.0,46.0,51.0,34.0,50.0
5,2015-02-08,75.0,80.0,46.0,51.0,34.0


In [3]:
# Janela móvel: pra cada ano de teste, treina um modelo novo só com os anos anteriores
semanas_testadas = []
modelos_por_ano = {}

for ano in ANOS_TESTE:
    treino, teste = separar_treino_teste(dengue, ano)

    # As 4 primeiras semanas de 2015 não têm todos os lags (não existe semana antes delas): saem do treino
    treino = treino.dropna(subset=FEATURES_LAGS)

    # fit = treinar: o modelo acha os pesos que melhor ligam os lags ao casos_est do treino
    modelo = LinearRegression()
    modelo.fit(treino[FEATURES_LAGS], treino["casos_est"])

    # predict = prever: aplica esses pesos nos lags das semanas de teste
    teste = teste.copy()
    teste["previsao_linear"] = modelo.predict(teste[FEATURES_LAGS])

    semanas_testadas.append(teste)
    modelos_por_ano[ano] = modelo

semanas_testadas = pd.concat(semanas_testadas)
print("Semanas testadas:", len(semanas_testadas))
print("Menor previsão:", round(semanas_testadas["previsao_linear"].min(), 1))

Semanas testadas: 391
Menor previsão: 2.3


In [4]:
# Baseline x regressão linear, com as mesmas regras (função do src/avaliacao.py)
erros_baseline = erros_por_ano(semanas_testadas, "previsao_ingenua")
erros_linear = erros_por_ano(semanas_testadas, "previsao_linear")

comparacao = pd.DataFrame({
    "ano": erros_baseline["ano"],
    "MAE_baseline": erros_baseline["MAE"],
    "MAE_linear": erros_linear["MAE"],
    "RMSE_baseline": erros_baseline["RMSE"],
    "RMSE_linear": erros_linear["RMSE"],
})
# Negativo = a regressão errou menos que o baseline
comparacao["diferenca_MAE"] = comparacao["MAE_linear"] - comparacao["MAE_baseline"]
comparacao.round(1)

,ano,MAE_baseline,MAE_linear,RMSE_baseline,RMSE_linear,diferenca_MAE
0,2019,18.7,19.2,28.2,27.2,0.5
1,2020,7.8,9.2,11.1,12.5,1.5
2,2021,7.9,9.0,12.7,13.7,1.1
3,2022,83.9,75.4,147.4,141.6,-8.5
4,2023,16.3,17.3,23.0,22.3,1.0
5,2024,30.4,30.5,40.7,42.6,0.1
6,2025,15.0,17.3,22.2,25.2,2.3
7,2026,61.9,57.2,91.7,86.6,-4.7
8,total,27.9,27.4,62.7,60.7,-0.6


In [5]:
# Os pesos que cada modelo aprendeu (um modelo por ano de teste)
# previsão = intercepto + peso1 x lag1 + peso2 x lag2 + peso3 x lag3 + peso4 x lag4
pesos = []
for ano in ANOS_TESTE:
    modelo = modelos_por_ano[ano]
    linha = {"ano_teste": ano, "intercepto": modelo.intercept_}
    for nome, peso in zip(FEATURES_LAGS, modelo.coef_):
        linha[nome] = peso
    pesos.append(linha)

pd.DataFrame(pesos).round(2)

,ano_teste,intercepto,casos_lag1,casos_lag2,casos_lag3,casos_lag4
0,2019,6.13,1.18,-0.22,0.17,-0.21
1,2020,6.10,1.17,-0.18,0.13,-0.20
2,2021,5.03,1.17,-0.18,0.13,-0.19
3,2022,4.42,1.15,-0.15,0.12,-0.18
4,2023,6.03,1.30,-0.46,0.18,-0.08
5,2024,6.06,1.30,-0.45,0.18,-0.08
6,2025,6.44,1.27,-0.41,0.16,-0.09
7,2026,6.56,1.26,-0.38,0.14,-0.08


### Conclusões do passo 1

- MAE 27,4 e RMSE 60,7, contra 27,9 e 62,7 do baseline: praticamente um empate (−2%).
- Ganha em 2022 (75,4 contra 83,9) e 2026 (57,2 contra 61,9), mas perde em 6 dos 8 anos, principalmente nos anos calmos.
- Os pesos são estáveis de um ano pro outro: o lag 1 fica entre 1,15 e 1,30 e o lag 2 é negativo. Na prática, o modelo pega a semana passada e acrescenta parte do quanto ela subiu: ele segue a tendência, e por isso ganha na subida de 2022.
- O intercepto soma uns 6 casos em toda semana, o que pesa nos anos com ~20 casos por semana. É a explicação mais provável pra derrota nos anos calmos (não conferi diretamente).

## Passo 2: + época do ano

O modelo do passo 1 não sabe em que ponto do ano está. Represento a semana do ano como um ponto num círculo, com seno e cosseno de `2π × semana / 52`, como o ponteiro de um relógio. Assim a semana 52 fica colada na semana 1, e a regressão consegue desenhar uma onda por ano. Com a semana como um número de 1 a 53, ela só conseguiria dizer "quanto maior a semana, mais casos".

O loop da janela móvel vira uma função, porque vai se repetir nos próximos passos.

In [6]:
import numpy as np

# Passo 2: a época do ano como um ponto num círculo (igual o ponteiro de um relógio)
# A semana vira um ângulo: semana 1 ≈ 0°, semana 13 = 90°, semana 26 = 180°, semana 39 = 270°, semana 52 = 360° (= 0° de novo)
# seno e cosseno são as coordenadas do ponteiro: assim a semana 52 fica colada na semana 1
dengue["semana_do_ano"] = dengue["SE"] % 100
angulo = 2 * np.pi * dengue["semana_do_ano"] / 52
dengue["semana_seno"] = np.sin(angulo)
dengue["semana_cosseno"] = np.cos(angulo)

FEATURES_EPOCA = ["semana_seno", "semana_cosseno"]

# Confere o "relógio" em algumas semanas: 1, 13, 26, 39, 52 e a semana 1 do ano seguinte
dengue[["SE", "semana_do_ano"] + FEATURES_EPOCA].iloc[[0, 12, 25, 38, 51, 52]].round(2)

,SE,semana_do_ano,semana_seno,semana_cosseno
0,201501,1,0.12,0.99
12,201513,13,1.00,-0.00
25,201526,26,-0.00,-1.00
38,201539,39,-1.00,-0.00
51,201552,52,0.00,1.00
52,201601,1,0.12,0.99


In [7]:
# O loop da janela móvel vai se repetir em todos os próximos passos, só mudando as features
# Por isso vira uma função: recebe a lista de features e o nome da coluna de previsão
# alvo_em_log=True (passo 3): treina prevendo log1p(casos) e volta pra casos com expm1 antes de devolver
def treinar_janela_movel(features, coluna_previsao, alvo_em_log=False):
    semanas = []
    modelos = {}
    for ano in ANOS_TESTE:
        treino, teste = separar_treino_teste(dengue, ano)
        treino = treino.dropna(subset=features)

        if alvo_em_log:
            alvo_treino = np.log1p(treino["casos_est"])
        else:
            alvo_treino = treino["casos_est"]

        modelo = LinearRegression()
        modelo.fit(treino[features], alvo_treino)

        teste = teste.copy()
        previsao = modelo.predict(teste[features])
        if alvo_em_log:
            # Volta pra escala real ANTES de medir o erro; senão o erro sairia em "log de casos"
            previsao = np.expm1(previsao)
        teste[coluna_previsao] = previsao

        semanas.append(teste)
        modelos[ano] = modelo

    return pd.concat(semanas), modelos


FEATURES_PASSO2 = FEATURES_LAGS + FEATURES_EPOCA
semanas_passo2, modelos_passo2 = treinar_janela_movel(FEATURES_PASSO2, "previsao_lags_epoca")

print("Semanas testadas:", len(semanas_passo2))
print("Menor previsão:", round(semanas_passo2["previsao_lags_epoca"].min(), 1))

Semanas testadas: 391
Menor previsão: 0.1


In [8]:
# Baseline x só lags (passo 1) x lags + época do ano (passo 2)
erros_passo2 = erros_por_ano(semanas_passo2, "previsao_lags_epoca")

comparacao_passo2 = pd.DataFrame({
    "ano": erros_baseline["ano"],
    "MAE_baseline": erros_baseline["MAE"],
    "MAE_lags": erros_linear["MAE"],
    "MAE_lags_epoca": erros_passo2["MAE"],
    "RMSE_baseline": erros_baseline["RMSE"],
    "RMSE_lags": erros_linear["RMSE"],
    "RMSE_lags_epoca": erros_passo2["RMSE"],
})
# Negativo = errou menos que o baseline
comparacao_passo2["diferenca_MAE_vs_baseline"] = comparacao_passo2["MAE_lags_epoca"] - comparacao_passo2["MAE_baseline"]
comparacao_passo2.round(1)

,ano,MAE_baseline,MAE_lags,MAE_lags_epoca,RMSE_baseline,RMSE_lags,RMSE_lags_epoca,diferenca_MAE_vs_baseline
0,2019,18.7,19.2,18.0,28.2,27.2,26.1,-0.7
1,2020,7.8,9.2,9.7,11.1,12.5,15.0,2.0
2,2021,7.9,9.0,11.3,12.7,13.7,15.2,3.4
3,2022,83.9,75.4,73.0,147.4,141.6,140.1,-11.0
4,2023,16.3,17.3,15.7,23.0,22.3,20.7,-0.6
5,2024,30.4,30.5,29.3,40.7,42.6,40.8,-1.1
6,2025,15.0,17.3,17.2,22.2,25.2,24.4,2.3
7,2026,61.9,57.2,54.9,91.7,86.6,84.7,-7.0
8,total,27.9,27.4,26.8,62.7,60.7,59.8,-1.2


In [9]:
# O que a época do ano acrescenta, em casos, no modelo mais recente (teste de 2026)
modelo_2026 = modelos_passo2[2026]
pesos_2026 = dict(zip(FEATURES_PASSO2, modelo_2026.coef_.round(2)))
print("Intercepto:", round(modelo_2026.intercept_, 1))
print("Pesos:", pesos_2026)

# A parte da época é uma onda: peso_seno x seno + peso_cosseno x cosseno
# Calcula essa onda em cada semana do ano pra ver quanto ela soma ou tira da previsão
semanas = pd.Series(range(1, 53))
angulo_semanas = 2 * np.pi * semanas / 52
efeito_epoca = pesos_2026["semana_seno"] * np.sin(angulo_semanas) + pesos_2026["semana_cosseno"] * np.cos(angulo_semanas)

print()
print("A época soma no máximo", round(efeito_epoca.max(), 1), "casos (semana", int(semanas[efeito_epoca.idxmax()]), ")")
print("e tira no máximo", round(-efeito_epoca.min(), 1), "casos (semana", int(semanas[efeito_epoca.idxmin()]), ")")

Intercepto: 8.1
Pesos: {'casos_lag1': np.float64(1.22), 'casos_lag2': np.float64(-0.37), 'casos_lag3': np.float64(0.13), 'casos_lag4': np.float64(-0.06), 'semana_seno': np.float64(12.81), 'semana_cosseno': np.float64(4.59)}

A época soma no máximo 13.6 casos (semana 10 )
e tira no máximo 13.6 casos (semana 36 )


### Conclusões do passo 2

- MAE 26,8 e RMSE 59,8: a primeira vitória sobre o baseline (−4,2% no MAE), ganhando em 5 de 8 anos, inclusive 2022 (−11 casos por semana) e 2026 (−7,0).
- Mas piora nos anos fracos: 2020 (+2,0), 2021 (+3,4) e 2025 (+2,3) ficam acima do baseline.
- O motivo é que, num modelo linear, a época entra como uma soma fixa: no máximo +13,6 casos na semana 10 (logo antes da subida) e −13,6 na semana 36. Isso ajuda a antecipar a subida num ano normal, mas em 2020 e 2021, com picos de 82 e 105 casos, é chutar um surto que não veio.
- A menor previsão foi 0,1, quase negativa.

## Passo 3: + alvo em log

Se a época entra como soma, o modelo pensa em casos absolutos. Em log, somar vira multiplicar: a época passaria a dizer "em março os casos crescem X%", o que deveria funcionar tanto em ano forte quanto em ano fraco. A EDA também mostrou uma cauda longa, que o log achata.

Uso o log no alvo e nos lags, pra os dois ficarem na mesma escala, e sempre volto pra casos com `expm1` antes de medir o erro.

In [10]:
# Passo 3: tudo em log (alvo e lags), pra o modelo pensar em proporção ("cresce X%") e não em casos somados
# log1p(x) = log(1 + x), igual na EDA; a época (seno e cosseno) continua igual, porque já vai de -1 a 1
FEATURES_LAGS_LOG = []
for k in range(1, 5):
    dengue[f"casos_lag{k}_log"] = np.log1p(dengue[f"casos_lag{k}"])
    FEATURES_LAGS_LOG.append(f"casos_lag{k}_log")

FEATURES_PASSO3 = FEATURES_LAGS_LOG + FEATURES_EPOCA
semanas_passo3, modelos_passo3 = treinar_janela_movel(FEATURES_PASSO3, "previsao_log", alvo_em_log=True)

print("Semanas testadas:", len(semanas_passo3))
# Com expm1 na volta, nenhuma previsão pode ser negativa
print("Menor previsão:", round(semanas_passo3["previsao_log"].min(), 2))

Semanas testadas: 391
Menor previsão: 0.62


In [11]:
# Baseline x passo 2 (lags + época, em casos) x passo 3 (lags + época, em log)
erros_passo3 = erros_por_ano(semanas_passo3, "previsao_log")

comparacao_passo3 = pd.DataFrame({
    "ano": erros_baseline["ano"],
    "MAE_baseline": erros_baseline["MAE"],
    "MAE_passo2": erros_passo2["MAE"],
    "MAE_passo3_log": erros_passo3["MAE"],
    "RMSE_baseline": erros_baseline["RMSE"],
    "RMSE_passo2": erros_passo2["RMSE"],
    "RMSE_passo3_log": erros_passo3["RMSE"],
})
comparacao_passo3.round(1)

,ano,MAE_baseline,MAE_passo2,MAE_passo3_log,RMSE_baseline,RMSE_passo2,RMSE_passo3_log
0,2019,18.7,18.0,20.0,28.2,26.1,30.5
1,2020,7.8,9.7,7.1,11.1,15.0,11.6
2,2021,7.9,11.3,8.0,12.7,15.2,13.1
3,2022,83.9,73.0,84.9,147.4,140.1,153.0
4,2023,16.3,15.7,15.2,23.0,20.7,21.3
5,2024,30.4,29.3,28.2,40.7,40.8,39.6
6,2025,15.0,17.2,14.6,22.2,24.4,20.4
7,2026,61.9,54.9,68.2,91.7,84.7,107.5
8,total,27.9,26.8,28.1,62.7,59.8,65.9


In [12]:
# Por que o log muda o comportamento? Os pesos do modelo mais recente (teste de 2026)
modelo_log_2026 = modelos_passo3[2026]
pesos_log = dict(zip(FEATURES_PASSO3, modelo_log_2026.coef_.round(3)))
print("Pesos:", pesos_log)

# Em log, os pesos dos lags somados dizem se o modelo segue a tendência (> 1) ou puxa de volta pro meio (< 1)
soma_lags = sum(pesos_log[nome] for nome in FEATURES_LAGS_LOG)
print("Soma dos pesos dos lags:", round(soma_lags, 3))

# Em log, a época vira um FATOR que multiplica a previsão: exp(peso_seno x seno + peso_cosseno x cosseno)
efeito_epoca_log = pesos_log["semana_seno"] * np.sin(angulo_semanas) + pesos_log["semana_cosseno"] * np.cos(angulo_semanas)
fator_epoca = np.exp(efeito_epoca_log)
print("A época multiplica a previsão por até", round(fator_epoca.max(), 2), "(semana", int(semanas[fator_epoca.idxmax()]), ")")
print("e por no mínimo", round(fator_epoca.min(), 2), "(semana", int(semanas[fator_epoca.idxmin()]), ")")

# Viés: erro médio (real - previsto). Positivo = o modelo prevê de menos, em média
vies_passo2 = (semanas_passo2["casos_est"] - semanas_passo2["previsao_lags_epoca"]).mean()
vies_passo3 = (semanas_passo3["casos_est"] - semanas_passo3["previsao_log"]).mean()
print()
print("Viés médio do passo 2:", round(vies_passo2, 1), "| do passo 3 (log):", round(vies_passo3, 1))

Pesos: {'casos_lag1_log': np.float64(0.807), 'casos_lag2_log': np.float64(0.056), 'casos_lag3_log': np.float64(-0.068), 'casos_lag4_log': np.float64(0.124), 'semana_seno': np.float64(0.194), 'semana_cosseno': np.float64(0.086)}
Soma dos pesos dos lags: 0.919
A época multiplica a previsão por até 1.24 (semana 10 )
e por no mínimo 0.81 (semana 36 )

Viés médio do passo 2: 3.4 | do passo 3 (log): 13.6


### Conclusões do passo 3

- MAE 28,1 e RMSE 65,9: pior que o baseline e que o passo 2.
- A hipótese acertou nos anos normais e fracos: em 2020 (7,1), 2023 (15,2), 2024 (28,2) e 2025 (14,6), o log foi o melhor de todos, e em 2021 (8,0) praticamente empatou com o baseline (7,9). A época virou mesmo um fator: ×1,24 na semana 10 e ×0,81 na semana 36.
- Mas errou muito nos surtos: 2022 (84,9) e 2026 (68,2), contra 73,0 e 54,9 do passo 2. A soma dos pesos dos lags em log é 0,919, menor que 1, então o modelo "puxa de volta pro meio" em vez de seguir a subida. E em média ele prevê 13,6 casos a menos que o real (o passo 2, 3,4), em parte porque a volta do log puxa a previsão pra perto da mediana.
- Como os surtos pesam muito no total, o log sai perdendo. O passo 4 vai em cima do passo 2.

## Passo 4: + clima

A grande pergunta do projeto: o clima acrescenta algo além dos casos recentes e da época do ano? Testo duas versões, uma coisa nova de cada vez: (A) a chuva acumulada das semanas t−1 a t−4 e t−5 a t−8, e (B) A mais a umidade média das semanas t−1 a t−4.

Todo clima entra com `shift(1)`, porque só vale o que já aconteceu até a semana anterior. As janelas são fixas (4 em 4 semanas) e foram escolhidas antes de ver o resultado, e não pelo lag que deu a maior correlação na EDA, pra não escolher olhando os anos de teste.

In [13]:
# Passo 4: o clima (ERA5), em cima do passo 2
# Pra prever a semana t só vale clima até a semana t-1: por isso tudo começa com shift(1)
# rolling(4) é uma janela que desliza: em cada linha, olha as 4 linhas anteriores (igual SUM(...) OVER (ROWS 3 PRECEDING) no SQL)
dengue["chuva_acum_1a4"] = dengue["chuva_mm"].shift(1).rolling(4).sum()       # chuva somada das semanas t-1 a t-4
dengue["chuva_acum_5a8"] = dengue["chuva_mm"].shift(5).rolling(4).sum()       # chuva somada das semanas t-5 a t-8
dengue["umidade_media_1a4"] = dengue["umidade_media"].shift(1).rolling(4).mean()  # umidade média das semanas t-1 a t-4

# Confere na mão numa linha qualquer (a linha 20): a soma das chuvas das linhas 16 a 19 tem que bater com chuva_acum_1a4
linha = 20
print("Chuva das linhas 16 a 19 somada na mão:", round(dengue["chuva_mm"].iloc[linha - 4:linha].sum(), 1))
print("chuva_acum_1a4 da linha 20:", round(dengue["chuva_acum_1a4"].iloc[linha], 1))
print("Chuva das linhas 12 a 15 somada na mão:", round(dengue["chuva_mm"].iloc[linha - 8:linha - 4].sum(), 1))
print("chuva_acum_5a8 da linha 20:", round(dengue["chuva_acum_5a8"].iloc[linha], 1))

Chuva das linhas 16 a 19 somada na mão: 106.2
chuva_acum_1a4 da linha 20: 106.2
Chuva das linhas 12 a 15 somada na mão: 156.0
chuva_acum_5a8 da linha 20: 156.0


In [14]:
# Dois modelos, uma coisa nova de cada vez:
# A = passo 2 + chuva acumulada | B = passo 2 + chuva acumulada + umidade
FEATURES_CHUVA = ["chuva_acum_1a4", "chuva_acum_5a8"]
FEATURES_PASSO4A = FEATURES_PASSO2 + FEATURES_CHUVA
FEATURES_PASSO4B = FEATURES_PASSO2 + FEATURES_CHUVA + ["umidade_media_1a4"]

semanas_passo4a, modelos_passo4a = treinar_janela_movel(FEATURES_PASSO4A, "previsao_chuva")
semanas_passo4b, modelos_passo4b = treinar_janela_movel(FEATURES_PASSO4B, "previsao_chuva_umidade")

erros_passo4a = erros_por_ano(semanas_passo4a, "previsao_chuva")
erros_passo4b = erros_por_ano(semanas_passo4b, "previsao_chuva_umidade")

comparacao_passo4 = pd.DataFrame({
    "ano": erros_baseline["ano"],
    "MAE_baseline": erros_baseline["MAE"],
    "MAE_passo2": erros_passo2["MAE"],
    "MAE_4A_chuva": erros_passo4a["MAE"],
    "MAE_4B_chuva_umidade": erros_passo4b["MAE"],
    "RMSE_passo2": erros_passo2["RMSE"],
    "RMSE_4A_chuva": erros_passo4a["RMSE"],
    "RMSE_4B_chuva_umidade": erros_passo4b["RMSE"],
})
comparacao_passo4.round(1)

,ano,MAE_baseline,MAE_passo2,MAE_4A_chuva,MAE_4B_chuva_umidade,RMSE_passo2,RMSE_4A_chuva,RMSE_4B_chuva_umidade
0,2019,18.7,18.0,18.1,18.2,26.1,25.6,25.5
1,2020,7.8,9.7,9.3,9.4,15.0,14.8,14.8
2,2021,7.9,11.3,11.3,11.3,15.2,15.2,15.4
3,2022,83.9,73.0,73.4,73.5,140.1,139.3,139.4
4,2023,16.3,15.7,15.4,15.3,20.7,22.1,22.1
5,2024,30.4,29.3,30.0,29.9,40.8,41.8,41.7
6,2025,15.0,17.2,16.9,16.9,24.4,24.4,24.4
7,2026,61.9,54.9,55.1,55.1,84.7,85.4,85.7
8,total,27.9,26.8,26.8,26.8,59.8,59.8,59.8


In [15]:
# Quanto o clima mexe na previsão? Os pesos das features de clima no modelo mais recente (teste de 2026)
modelo_4b_2026 = modelos_passo4b[2026]
pesos_4b = dict(zip(FEATURES_PASSO4B, modelo_4b_2026.coef_))

for nome in FEATURES_CHUVA + ["umidade_media_1a4"]:
    print(nome, "->", round(pesos_4b[nome], 3))

# Traduzindo pra casos: 100 mm a mais de chuva nas últimas 4 semanas mudam a previsão em quantos casos?
print()
print("100 mm a mais nas semanas t-1 a t-4:", round(100 * pesos_4b["chuva_acum_1a4"], 1), "casos")
print("100 mm a mais nas semanas t-5 a t-8:", round(100 * pesos_4b["chuva_acum_5a8"], 1), "casos")

print()
print("Menor previsão do 4A:", round(semanas_passo4a["previsao_chuva"].min(), 1))
print("Menor previsão do 4B:", round(semanas_passo4b["previsao_chuva_umidade"].min(), 1))

chuva_acum_1a4 -> 0.063
chuva_acum_5a8 -> -0.058
umidade_media_1a4 -> -0.242

100 mm a mais nas semanas t-1 a t-4: 6.3 casos
100 mm a mais nas semanas t-5 a t-8: -5.8 casos

Menor previsão do 4A: -0.9
Menor previsão do 4B: -4.6


### Conclusões do passo 4

- O clima não acrescentou nada: 4A e 4B dão MAE 26,8, igual ao passo 2, e nenhum ano muda mais que 0,7 caso por semana.
- No modelo mais recente, 100 mm a mais de chuva nas semanas t−1 a t−4 somam 6,3 casos, e nas semanas t−5 a t−8 tiram 5,8. Os dois praticamente se cancelam.
- A umidade ficou com peso negativo (−0,24 caso por ponto), o contrário do esperado. O mais provável é colinearidade com o seno e o cosseno: as duas dizem "é época de chuva", e o modelo divide o peso entre elas (não conferi).
- Com o clima apareceram previsões negativas (−0,9 no 4A e −4,6 no 4B).
- Isso confirma a EDA por outro caminho: pra prever 1 semana à frente, o clima semanal do ERA5 não traz informação além dos casos recentes e da época do ano. Não quer dizer que o clima não influencia a dengue; quer dizer que, nesse horizonte, a influência já aparece nos casos recentes. Os limites continuam: ERA5 com grade de ~25 km, chuva com correlação de 0,71 com a estação, modelo linear.

## Passo 5: um modelo não linear

Um Gradient Boosting (árvores em sequência, cada uma corrigindo o erro das anteriores) pode captar combinações que a regressão linear não enxerga, como "época de subida e casos crescendo rápido". O problema é o teto das árvores: elas não preveem acima do maior valor do treino. Pra contornar, o alvo vira a taxa de crescimento em relação à semana anterior, `log(casos + 1) − log(casos da semana anterior + 1)`, e a previsão volta pra casos multiplicando a semana anterior por esse crescimento.

Mesmas features do passo 2, sem o clima. Parâmetros padrão e `random_state` fixo, sem ajustar nada olhando o teste.

In [16]:
from sklearn.ensemble import HistGradientBoostingRegressor

# Passo 5: Gradient Boosting (árvores em sequência, cada uma corrigindo o erro das anteriores)
# Árvores não preveem acima do maior valor do treino (em 2022 o treino tinha no máximo 620 casos/semana).
# Por isso o alvo vira a TAXA de crescimento em relação à semana anterior, e não o número de casos:
#   taxa = log(casos + 1) - log(casos da semana anterior + 1)
# Depois volta pra casos: previsão = (casos da semana anterior + 1) x exp(taxa prevista) - 1
dengue["taxa_crescimento"] = np.log1p(dengue["casos_est"]) - np.log1p(dengue["casos_lag1"])


def treinar_boosting_taxa(features, coluna_previsao):
    semanas = []
    for ano in ANOS_TESTE:
        treino, teste = separar_treino_teste(dengue, ano)
        treino = treino.dropna(subset=features)

        # random_state fixo (a "seed") pra o resultado sair igual toda vez; parâmetros padrão, sem ajustar pelo teste
        modelo = HistGradientBoostingRegressor(random_state=42)
        modelo.fit(treino[features], treino["taxa_crescimento"])

        teste = teste.copy()
        taxa_prevista = modelo.predict(teste[features])
        # log1p e expm1 são o caminho de ida e volta: log(x + 1) e exp(x) - 1
        teste[coluna_previsao] = np.expm1(np.log1p(teste["casos_lag1"]) + taxa_prevista)
        teste["taxa_prevista"] = taxa_prevista
        semanas.append(teste)

    return pd.concat(semanas)


# Mesmas features do passo 2 (lags + época), sem o clima, que não ajudou no passo 4
semanas_passo5 = treinar_boosting_taxa(FEATURES_PASSO2, "previsao_boosting")

print("Semanas testadas:", len(semanas_passo5))
print("Menor previsão:", round(semanas_passo5["previsao_boosting"].min(), 1))
print("Maior previsão:", round(semanas_passo5["previsao_boosting"].max(), 1))

Semanas testadas: 391
Menor previsão: 0.2
Maior previsão: 1488.2


In [17]:
# Baseline x passo 2 (melhor até aqui) x passo 5 (boosting na taxa)
erros_passo5 = erros_por_ano(semanas_passo5, "previsao_boosting")

comparacao_passo5 = pd.DataFrame({
    "ano": erros_baseline["ano"],
    "MAE_baseline": erros_baseline["MAE"],
    "MAE_passo2": erros_passo2["MAE"],
    "MAE_passo5_boosting": erros_passo5["MAE"],
    "RMSE_baseline": erros_baseline["RMSE"],
    "RMSE_passo2": erros_passo2["RMSE"],
    "RMSE_passo5_boosting": erros_passo5["RMSE"],
})
comparacao_passo5.round(1)

,ano,MAE_baseline,MAE_passo2,MAE_passo5_boosting,RMSE_baseline,RMSE_passo2,RMSE_passo5_boosting
0,2019,18.7,18.0,18.7,28.2,26.1,27.2
1,2020,7.8,9.7,8.2,11.1,15.0,11.6
2,2021,7.9,11.3,8.6,12.7,15.2,14.6
3,2022,83.9,73.0,68.8,147.4,140.1,137.6
4,2023,16.3,15.7,18.8,23.0,20.7,29.9
5,2024,30.4,29.3,27.8,40.7,40.8,41.0
6,2025,15.0,17.2,20.8,22.2,24.4,28.1
7,2026,61.9,54.9,70.9,91.7,84.7,105.3
8,total,27.9,26.8,27.4,62.7,59.8,61.8


In [18]:
# O teto mudou de lugar: agora vale pra TAXA. Qual a maior taxa que cada treino viu, e qual o modelo previu no teste?
# taxa 0,5 = cresceu ~65% numa semana (exp(0,5) = 1,65)
limites = []
for ano in ANOS_TESTE:
    treino, teste = separar_treino_teste(dengue, ano)
    do_ano = semanas_passo5[semanas_passo5["ano"] == ano]
    limites.append({
        "ano_teste": ano,
        "maior_taxa_no_treino": treino["taxa_crescimento"].max(),
        "maior_taxa_prevista": do_ano["taxa_prevista"].max(),
        "maior_taxa_real_no_teste": do_ano["taxa_crescimento"].max(),
    })
# to_string mostra a tabela inteira, sem cortar colunas com "..."
print(pd.DataFrame(limites).round(2).to_string(index=False))

# O pico de 2022 (semana de 17/04): o real, o baseline e o boosting
pico_2022 = semanas_passo5[semanas_passo5["data_iniSE"] == "2022-04-17"]
print()
print("Pico de 2022 -> real:", pico_2022["casos_est"].iloc[0],
      "| baseline:", round(pico_2022["previsao_ingenua"].iloc[0]),
      "| boosting:", round(pico_2022["previsao_boosting"].iloc[0]))

# Pra que lado erra: positivo = previu de menos
semanas_passo5["erro_boosting"] = semanas_passo5["casos_est"] - semanas_passo5["previsao_boosting"]
semanas_passo5["momento"] = np.where(semanas_passo5["casos_est"] > semanas_passo5["casos_lag1"], "subindo", "caindo ou parado")
print()
print("Erro médio do boosting por momento:")
print(semanas_passo5.groupby("momento")["erro_boosting"].mean().round(1))

 ano_teste  maior_taxa_no_treino  maior_taxa_prevista  maior_taxa_real_no_teste
      2019                  0.89                 0.42                      0.51
      2020                  0.89                 0.58                      0.69
      2021                  0.89                 0.50                      1.32
      2022                  1.32                 0.37                      3.78
      2023                  3.78                 0.68                      1.35
      2024                  3.78                 0.38                      0.85
      2025                  3.78                 0.40                      0.43
      2026                  3.78                 0.35                      0.33

Pico de 2022 -> real: 1581.0 | baseline: 1047 | boosting: 951

Erro médio do boosting por momento:
momento
caindo ou parado   -12.8
subindo             26.2
Name: erro_boosting, dtype: float64


### Conclusões do passo 5

- MAE 27,4 e RMSE 61,8: um pouco melhor que o baseline nos dois, mas pior que o passo 2.
- É o modelo mais instável: o melhor de todos em 2022 (68,8) e 2024 (27,8), bom em 2020 e 2021, mas ruim em 2023 (18,8), 2025 (20,8) e principalmente 2026 (70,9, com RMSE de 105,3).
- O teto não sumiu, mudou de lugar: o modelo nunca previu uma taxa acima de 0,68, mesmo o treino tendo taxas de até 3,78. Na prática ele freia as subidas: no pico de 2022 previu 951 (o baseline previu 1.047 e o real foi 1.581).
- A taxa de 3,78 vem de uma semana com 0 casos (26/12/2021) seguida de uma com 43 (02/01/2022). Tem cara de artefato da virada de ano na notificação, mas não conferi.
- Com 200 a 570 semanas de treino, o boosting tende a decorar padrões que não se repetem. A regressão linear, com 7 números, é mais estável.

## Resumo dos passos

| passo | modelo | MAE | RMSE |
|---|---|---|---|
| — | baseline (semana anterior) | 27,9 | 62,7 |
| 1 | linear, lags | 27,4 | 60,7 |
| **2** | **linear, lags + época** | **26,8** | **59,8** |
| 3 | linear em log | 28,1 | 65,9 |
| 4 | linear + clima | 26,8 | 59,8 |
| 5 | boosting na taxa | 27,4 | 61,8 |

O melhor é o passo 2: o melhor total, estável entre os anos e explicável peso a peso. Ele bate o baseline por 4,2% no MAE, e o ganho fica concentrado nos surtos.

A célula abaixo salva essa tabela em `models/ablacao.csv`, com os números cheios. O dashboard lê esse arquivo: assim nenhum número do painel é digitado à mão, e se os dados mudarem é só rodar o notebook de novo.

In [19]:
# Tabela da ablação: o total de cada passo, gerada daqui pra o dashboard não ter número digitado à mão
passos = [
    ("baseline", "semana que vem = esta semana", erros_baseline),
    ("passo 1", "regressão linear, lags 1 a 4", erros_linear),
    ("passo 2", "passo 1 + época do ano (escolhido)", erros_passo2),
    ("passo 3", "passo 2 em log", erros_passo3),
    ("passo 4A", "passo 2 + chuva acumulada", erros_passo4a),
    ("passo 4B", "passo 4A + umidade", erros_passo4b),
    ("passo 5", "Gradient Boosting na taxa de crescimento", erros_passo5),
]

linhas = []
for passo, descricao, erros in passos:
    total = erros[erros["ano"] == "total"].iloc[0]
    linhas.append({"passo": passo, "modelo": descricao, "MAE": round(total["MAE"], 2), "RMSE": round(total["RMSE"], 2)})

ablacao = pd.DataFrame(linhas)
ablacao.to_csv("../models/ablacao.csv", index=False)
ablacao

,passo,modelo,MAE,RMSE
0,baseline,semana que vem = esta semana,27.95,62.69
1,passo 1,"regressão linear, lags 1 a 4",27.38,60.70
2,passo 2,passo 1 + época do ano (escolhido),26.76,59.83
3,passo 3,passo 2 em log,28.09,65.88
4,passo 4A,passo 2 + chuva acumulada,26.79,59.78
5,passo 4B,passo 4A + umidade,26.79,59.81
6,passo 5,Gradient Boosting na taxa de crescimento,27.42,61.81


## Modelo final

Até aqui, cada ano de teste teve o seu modelo, treinado só com o passado, porque o objetivo era avaliar. Pra usar no dashboard, treino um modelo só, com a receita do passo 2 e todas as semanas fechadas. Ele não tem nota própria (não sobra semana fechada pra testar); a nota dele é a da receita na janela móvel: MAE 26,8.

As features passam a vir de `src/features.py`, a mesma função que o dashboard vai usar, pra treino e previsão calcularem exatamente igual. O modelo é salvo com `joblib`, junto de um JSON com a "ficha" dele (features, pesos, período de treino e avaliação).

In [20]:
# Modelo final: a receita do passo 2 (regressão linear, lags 1-4 + seno/cosseno da semana)
# As features agora vêm do src/features.py, a mesma função que o dashboard vai usar
from src.features import FEATURES_MODELO, criar_features

dados_modelo = criar_features(dengue[["data_iniSE", "SE", "ano", "casos_est", "semana_aberta"]])

# Confere que a função do src/ calcula exatamente o que o notebook calculou nos passos 1 e 2
for coluna in FEATURES_MODELO:
    iguais = dados_modelo[coluna].equals(dengue[coluna])
    print(coluna, "igual ao notebook:", iguais)

# Treina com TODAS as semanas fechadas (sem as 4 primeiras de 2015, que não têm todos os lags)
treino_final = dados_modelo[~dados_modelo["semana_aberta"]].dropna(subset=FEATURES_MODELO)

modelo_final = LinearRegression()
modelo_final.fit(treino_final[FEATURES_MODELO], treino_final["casos_est"])

print()
print("Semanas no treino:", len(treino_final))
print("De", treino_final["data_iniSE"].min().date(), "a", treino_final["data_iniSE"].max().date())

casos_lag1 igual ao notebook: True
casos_lag2 igual ao notebook: True
casos_lag3 igual ao notebook: True
casos_lag4 igual ao notebook: True
semana_seno igual ao notebook: True
semana_cosseno igual ao notebook: True

Semanas no treino: 595
De 2015-02-01 a 2026-06-21


### Faixa de 80%

Só o número da previsão ("19 casos") não diz o quanto dá pra confiar nele. Pra mostrar uma faixa no dashboard, uso os erros do passo 2 na janela móvel: calculo o erro em escala log (`log1p(real) − log1p(previsto)`) e pego os quantis de 10% e 90%. Em log, a faixa fica proporcional à previsão: estreita em semana calma e larga em surto, igual ao erro.

Depois confiro em quantas semanas de teste o real caiu dentro da faixa. As previsões da janela móvel ficam salvas em `models/backtest_modelo_final.csv`, que a aba de avaliação do dashboard usa.

In [21]:
# Faixa de incerteza (pro dashboard), tirada dos erros da janela móvel da receita final (passo 2)
from src.avaliacao import aplicar_faixa, erro_proporcional

# Erro proporcional de cada semana de teste: quanto o real ficou acima/abaixo do previsto, em escala log
erros_prop = erro_proporcional(semanas_passo2["casos_est"], semanas_passo2["previsao_lags_epoca"])

# Quantis: 10% dos erros ficam abaixo do quantil_10 e 10% acima do quantil_90; entre os dois ficam 80%
# (a mediana é o quantil de 50%)
quantil_10 = erros_prop.quantile(0.10)
quantil_90 = erros_prop.quantile(0.90)
print("Quantis do erro em log:", round(quantil_10, 3), "e", round(quantil_90, 3))
print("Em fator: de", round(np.exp(quantil_10), 2), "x a", round(np.exp(quantil_90), 2), "x a previsão")

# Confere a promessa: em quantas semanas de teste o real caiu dentro da faixa?
faixa_baixa, faixa_alta = aplicar_faixa(semanas_passo2["previsao_lags_epoca"], quantil_10, quantil_90)
dentro = (semanas_passo2["casos_est"] >= faixa_baixa) & (semanas_passo2["casos_est"] <= faixa_alta)
print("Cobertura total:", round(dentro.mean() * 100, 1), "%")
print("Cobertura por ano (%):", (dentro.groupby(semanas_passo2["ano"]).mean() * 100).round(0).to_dict())

# Salva as previsões da janela móvel: a faixa sai delas, e a aba de avaliação do dashboard vai usar esse arquivo
backtest = semanas_passo2[["data_iniSE", "SE", "ano", "casos_est", "previsao_ingenua", "previsao_lags_epoca"]].copy()
backtest = backtest.rename(columns={"previsao_ingenua": "previsao_baseline", "previsao_lags_epoca": "previsao_modelo"})
backtest["previsao_modelo"] = backtest["previsao_modelo"].round(2)
backtest.to_csv("../models/backtest_modelo_final.csv", index=False)
print()
print("Backtest salvo:", len(backtest), "semanas")

Quantis do erro em log: -0.616 e 0.442
Em fator: de 0.54 x a 1.56 x a previsão
Cobertura total: 79.8 %
Cobertura por ano (%): {2019: 85.0, 2020: 70.0, 2021: 56.0, 2022: 69.0, 2023: 87.0, 2024: 87.0, 2025: 96.0, 2026: 100.0}

Backtest salvo: 391 semanas


In [22]:
import json

import joblib
import sklearn

CAMINHO_MODELO = "../models/modelo_final.joblib"
CAMINHO_INFO = "../models/modelo_final.json"

# joblib grava o objeto do modelo inteiro num arquivo (no dashboard: joblib.load + predict)
joblib.dump(modelo_final, CAMINHO_MODELO)

# O JSON é a "ficha" do modelo, legível por qualquer um (tipo um package.json)
# float(...) converte o número do numpy pra float normal do Python, que o json sabe gravar
# Sem data de criação de propósito: assim o arquivo só muda quando o modelo ou os dados mudam
total_passo2 = erros_passo2[erros_passo2["ano"] == "total"].iloc[0]
total_baseline = erros_baseline[erros_baseline["ano"] == "total"].iloc[0]

pesos_final = {}
for nome, peso in zip(FEATURES_MODELO, modelo_final.coef_):
    pesos_final[nome] = float(peso)

info_modelo = {
    "modelo": "LinearRegression (scikit-learn)",
    "alvo": "casos_est da semana seguinte",
    "features": FEATURES_MODELO,
    "intercepto": float(modelo_final.intercept_),
    "pesos": pesos_final,
    "treino": {
        "primeira_semana": str(treino_final["data_iniSE"].min().date()),
        "ultima_semana": str(treino_final["data_iniSE"].max().date()),
        "semanas": len(treino_final),
    },
    "avaliacao_janela_movel_2019_2026": {
        "MAE_modelo": float(total_passo2["MAE"]),
        "RMSE_modelo": float(total_passo2["RMSE"]),
        "MAE_baseline": float(total_baseline["MAE"]),
        "RMSE_baseline": float(total_baseline["RMSE"]),
    },
    # Faixa de 80%: quantis do erro proporcional (em log) da janela móvel; o dashboard usa com aplicar_faixa
    "faixa_80": {
        "quantil_10_log": float(quantil_10),
        "quantil_90_log": float(quantil_90),
        "cobertura_janela_movel": float(dentro.mean()),
    },
    "versao_scikit_learn": sklearn.__version__,
}

# indent=2 deixa o arquivo formatado; ensure_ascii=False mantém os acentos
with open(CAMINHO_INFO, "w", encoding="utf-8") as arquivo:
    json.dump(info_modelo, arquivo, indent=2, ensure_ascii=False)

print("Salvo:", CAMINHO_MODELO, "e", CAMINHO_INFO)

Salvo: ../models/modelo_final.joblib e ../models/modelo_final.json


In [23]:
# Confere: o modelo relido do arquivo tem que prever exatamente igual ao da memória
modelo_relido = joblib.load(CAMINHO_MODELO)

semanas_com_features = dados_modelo.dropna(subset=FEATURES_MODELO)
previsao_memoria = modelo_final.predict(semanas_com_features[FEATURES_MODELO])
previsao_relido = modelo_relido.predict(semanas_com_features[FEATURES_MODELO])
print("Previsões iguais depois de reler:", np.allclose(previsao_memoria, previsao_relido))

# E a ficha relida do JSON
with open(CAMINHO_INFO, encoding="utf-8") as arquivo:
    info_relida = json.load(arquivo)

print("Intercepto:", round(info_relida["intercepto"], 2))
for nome, peso in info_relida["pesos"].items():
    print(" ", nome, "->", round(peso, 3))

Previsões iguais depois de reler: True
Intercepto: 9.08
  casos_lag1 -> 1.224
  casos_lag2 -> -0.332
  casos_lag3 -> 0.092
  casos_lag4 -> -0.064
  semana_seno -> 14.168
  semana_cosseno -> 3.582


### Conclusões

- O modelo final é uma regressão linear com os casos das 4 semanas anteriores e a época do ano, treinada com 595 semanas fechadas (01/02/2015 a 21/06/2026). A função do `src/features.py` calcula as features exatamente igual ao notebook.
- Ficou salvo em `models/modelo_final.joblib`, com a ficha em `models/modelo_final.json`, e o modelo relido do arquivo prevê igual ao da memória.
- Os pesos são parecidos com os dos modelos da avaliação (lag 1 ≈ 1,22, lag 2 negativo, época com ±15 casos): treinar com mais dados não mudou o comportamento.
- A faixa de 80% vai de 0,54× a 1,56× a previsão, e o real caiu dentro dela em 79,8% das semanas de teste. Mas isso muda muito de um ano pro outro: 56% em 2021 e 69% em 2022, contra 96% em 2025. Nos anos fora do comum, ela acerta menos.
- A conclusão da etapa: a dengue da semana que vem em Teresina é prevista principalmente pelos casos das últimas semanas e pela época do ano. O clima não acrescentou, o log ajudou só nos anos calmos e o modelo mais complexo não compensou. O ganho sobre o baseline é pequeno (4,2%), mas é medido de forma honesta, nos mesmos anos, incluindo o surto de 2022.